In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pycbc
import pycbc.waveform
import scipy
import lal as _lal

# Fourier Transform definitions

Below is a quick outline of the definitions of the discrete fourier transform that we will use, it is useful to keep track of factors before some of these terms to keep everthing consistent later on. See http://arxiv.org/abs/0911.3820 for more details.

If we have a time series a(t) sampled at intervals $\Delta t = 1/{2f_{\rm{Ny}}}$, where $f_{\rm{Ny}}$ is the Nyquist frequency for observation of length $T$. Total samples are then $N = T/\Delta t = 2Tf_{\rm{Ny}}$.

Start with some definitions, The fourier transform,
$$\tilde{a}(f) = \int^{\infty}_{-\infty}a(t)e^{2 \pi i f t} dt$$ 

The data points at a time $t_j$ and frequency $f_k$ are then,
$$ a(t_j) = a(j\Delta t) = a_j $$
$$ \tilde{a}(f_k) = \tilde{a}(k/T) = \Delta t \; \tilde{a}_k$$

The discrete Fourier series is then defined as,

$$\tilde{a}_k = \sum_j a_j e^{-2 \pi i j k / N}$$
$$a_j = \frac{1}{N}\sum_k a_k e^{2 \pi i j k / N}$$


# White time domain data

We start by loading in some data. The file **white_noise_ts.txt** in the 'data' folder contains the time series for some white noise (noise with equal intensity at different frequencies). There are two columns: the time corresponding to the strain measurement, and the strain.

In [ ]:
with open('./data/DET/DET_white_data_ts.txt', 'r') as f:
    times, strain = np.loadtxt(f).T

In [ ]:
duration = times[-1] - times[0]
Nt = len(times)
delta_t = duration / Nt

In [ ]:
# we can put this into a pycbc time series object which allows for some easy to use functions later on
data_t_pycbc = pycbc.types.TimeSeries(strain, delta_t = delta_t)

In [ ]:
fig, ax = plt.subplots()
ax.plot(data_t_pycbc.sample_times, data_t_pycbc.data)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Strain')

## Frequency domain signals

When taking a fourier transform of the data one should window the timeseries first as the Fourier series is defined for periodic functions, therefore we can end up with a discontinuity when the start and end edges are joined to make this periodic. Window functions smoothly makes the edge of the timeseries go to zero such that there are not discontinuities. 

In [ ]:
hann = scipy.signal.windows.hann(Nt)
tukey = scipy.signal.windows.tukey(Nt)
rect = np.ones(Nt)
rect[0] = 0
rect[-1] = 0

In [ ]:
fig, ax = plt.subplots()
ax.plot(times, hann, label='Hann')
ax.plot(times, tukey, label='Tukey')
ax.plot(times, rect, label='Rect')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Window Function')

In [ ]:
# compute the fast fourier transform of the data
data_f = np.fft.rfft(strain*hann)
# the frequencies corresponding to the fourier transform
data_f_frequencies = np.fft.rfftfreq(Nt, delta_t)

In [ ]:
fig, ax = plt.subplots()
ax.plot(data_f_frequencies, np.real(data_f), label='real')
ax.plot(data_f_frequencies, np.imag(data_f), label='imag')
ax.legend()
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Frequency domain strain")

We can also have a look at the power spectrum of this section of data. It is not entirely clear what windowing has done to this data as it is white Gaussian noise. See the next section for how this can have a large impact on the data.

In [ ]:
fig, ax = plt.subplots()
ax.plot( np.abs(data_f)**2)
ax.legend()
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Power spectrum")

### Brief aside on the definition of the Power Spectral Density (PSD)

Another useful quantity is the autocorrelation of the noise $R(\tau)$ which is defined as
$$ R(\tau) = \left<n(t)n(t+\tau)\right> = \int n(t) n(t+\tau)dt $$

This measures the timescales over which the noise is fluctuating, below you can see that there is higher correlation at lower timescales i.e. higher frequencies.

In [ ]:
ac_noise = np.random.normal(size = 1024) + np.sin(np.linspace(0,2,1024)) # add sinusoid to include some long duration flucuations
acorr = np.correlate(ac_noise, ac_noise, mode="full")

In [ ]:
fig, ax = plt.subplots()
ax.plot(acorr[-len(ac_noise) + 1:])
ax.set_xlabel("timescale (tau))")
ax.set_ylabel("Autocorrlelation (R(tau))")

We can then define the one sided noise power spectral density as the Fourier transform of the autocorrelation of the noise, i.e.

$$ S(f) = 2\int^{\infty}_{-\infty} R(\tau) e^{2 \pi i f \tau} dt\tau $$

This describes the statistics of the noise rather than one realisation like the power spectrum does.

# Now lets look at some real LIGO noise

In [ ]:
with open('./data/ligo_noise_ts.txt', 'r') as f:
    ligo_times, ligo_strain = np.loadtxt(f).T
    
ligo_times, ligo_strain = ligo_times[:Nt], ligo_strain[:Nt] # take only the first Nt samples from this to match window
# compute the fast fourier transform of the data (the data is hann windowed)
ligo_data_f = np.fft.rfft(ligo_strain*hann)
# the frequencies corresponding to the fourier transform
ligo_data_f_frequencies = np.fft.rfftfreq(Nt, delta_t)

In [ ]:
fig, ax = plt.subplots()
ax.plot(ligo_data_f_frequencies, np.abs(ligo_data_f)**2)
ax.set_xlim(1, 1000)
ax.set_ylim(1e-44, 1e-30)
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Power spectrum")

A Power Spectral Density (PSD) is a measure of the noise, therefore using the Power spectrum defined above is a bad approximation to the noise. Instead we can use the welch method, which computes many power spectra and takes the average.

We can also see how using different window function defined above have an effect on the psd. 

In [ ]:
nperseg = 1024
psd_hann = scipy.signal.windows.hann(nperseg)
psd_tukey = scipy.signal.windows.tukey(nperseg)
psd_rect = np.ones(nperseg)
psd_rect[0] = 0
psd_rect[-1] = 0

psd = scipy.signal.welch(ligo_strain, fs=1/delta_t, nperseg=nperseg, window=psd_rect)
psd_hann = scipy.signal.welch(ligo_strain, fs=1/delta_t, nperseg=nperseg, window=psd_hann)
psd_tukey = scipy.signal.welch(ligo_strain, fs=1/delta_t, nperseg=nperseg, window=psd_tukey)

In [ ]:
fig,ax = plt.subplots()
ax.plot(psd[0], psd[1], label="rect")
ax.plot(psd_hann[0], psd_hann[1], label="hann")
ax.plot(psd_tukey[0], psd_tukey[1], label="tukey")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim([1, 1000])
ax.set_ylim([0.05e-47, 2e-37])
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("PSD [Hz^-1]")

We can see that the rectangular window (which is equivalent to no window at all!) is wildly different from the others. This is because of the discontinuity at the boundary. The discontinuity causes spectral leakage, which takes power from the peaks and smears it out across different frequencies.

This is related to the fourier transform of the heaviside step function, because the multiplication by a window in the time domain is equivalent to the convolution of the FT of the data and window functions in the frequency domain:

By plotting the fourier transform of the window functions we can see the frequency response of the window that causes spectral leakage:

In [ ]:
f = np.fft.rfftfreq(Nt, d=delta_t)
plt.semilogy(f, np.fft.rfft(hann)**2, label='Hann')
plt.semilogy(f, np.fft.rfft(tukey)**2, label='Tukey')
plt.semilogy(f, np.fft.rfft(rect)**2, label='Rect')
plt.legend()
plt.xlabel('Frequency [Hz]')
plt.ylabel('$|\~{w}|^2$')
plt.xlim([0,20]);

Although all the windows have some leakage, it rapidly dies off for the Tukey and Hann windows, but stays constant for the rectangular one, meaning that power is spread out across all frequencies. This means that our estimate of the PSD will be distorted if we use the rectangular window, so instead we must use one of the smooth ones. Hann and Tukey are common choices but there are many more. 

## Generating a gravitational wave template

While this isn't strictly signal processing, you will need to generate signal templates for both the detection and parameter estimation routes of the lab. This can be done straightforwardly using the [PyCBC software](https://pycbc.org/).

In [ ]:
sampling_frequency = 1024 # Hz
duration = 1 # s
dt = 1./sampling_frequency # separation of samples in seconds
df = 1/duration # size of frequency bins

### Time domain waveforms

In [ ]:
# Generate the time domain signal (template) for a binary black hole merger between a 36 solar mass black hole and a 30 solar mass black hole at a distance of 4000 Mpc

hplus, hcross = pycbc.waveform.get_td_waveform(mass1=36,mass2=30,distance=4000,approximant='IMRPhenomPv2',f_lower=30, delta_t=dt)
print('PyCBC waveform has length ',len(hplus))
times = hplus.sample_times

In [ ]:
fig,ax = plt.subplots()
ax.plot(times,hplus)
ax.set_xlabel('time (seconds)')
ax.set_ylabel('strain')

### Frequency domain waveforms

In [ ]:
# Generate the frequency domain signal (template) for a binary black hole merger between a 36 solar mass black hole and a 30 solar mass black hole at a distance of 4000 Mpc

hplusf, hcrossf = pycbc.waveform.get_fd_waveform(mass1=36,mass2=30,distance=4000,approximant='IMRPhenomPv2',f_lower=30, delta_f=df)
print('PyCBC waveform has length ',len(hplusf))
freqs = hplusf.sample_frequencies

In [ ]:
fig,ax = plt.subplots()
ax.plot(freqs,hplusf.data)
ax.set_xlabel('frequency (Hz)')
ax.set_ylabel('strain')

**Q: What effect does changing the masses of the event have on the signal produced?**